# Judge

Consumes `unscored/` milestone snapshots written by `generation.ipynb` and writes `scored/` files. Base `perf/` materialization and edited-method `perf/` rebuilding are explicit cells around the worker.

## Environment

In [ ]:
import sys
from datetime import datetime
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / "src").exists() and (path / "notebook").exists():
            return path
    raise RuntimeError(f"Could not find project root from {start}")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

LOG_DIR = PROJECT_ROOT / "data" / "evaluation" / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = LOG_DIR / f"judge-{datetime.now():%Y-%m-%d-%H-%M-%S}.log"

## Evaluation Config

In [ ]:
import copy

import yaml

def resolve_run_indices(n_runs: int, run_start: int = 1, run_count: int | None = None) -> list[int]:
    if n_runs <= 0:
        raise ValueError("n_runs must be positive")
    if run_start <= 0:
        raise ValueError("run_start must be positive")

    if run_count is None:
        run_end = n_runs
    else:
        run_count = int(run_count)
        if run_count <= 0:
            raise ValueError("run_count must be positive when set")
        run_end = run_start + run_count - 1

    if run_end > n_runs:
        raise ValueError(
            f"run_start/run_count select runs {run_start}..{run_end}, "
            f"but n_runs={n_runs}"
        )

    return list(range(run_start, run_end + 1))


EVAL_CONFIG_PATH = PROJECT_ROOT / "src" / "evaluate" / "config.yaml"
with open(EVAL_CONFIG_PATH, encoding="utf-8") as file:
    eval_config = yaml.safe_load(file)

editing_methods = list(eval_config["editing_methods"])
model_key = eval_config["model_key"]
result_names = list(eval_config["result_names"])
eval_milestones = list(eval_config["eval_milestones"])
n_runs = int(eval_config["n_runs"])
run_start = int(eval_config.get("run_start", 1))
raw_run_count = eval_config.get("run_count")
run_count = None if raw_run_count is None else int(raw_run_count)
run_indices = resolve_run_indices(n_runs, run_start, run_count)
judge_repeats = int(eval_config["judge"]["repeats"])
base_seed = int(eval_config["base_seed"])

judge_config = copy.deepcopy(eval_config["judge"])
judge_config["log_path"] = LOG_PATH
judge_config.setdefault("model_config", {})["seed"] = base_seed

worker_config = copy.deepcopy(eval_config["worker"])

## Materialize Base Perf

In [ ]:
from src.evaluate.judge import materialize_base_perf_results

materialize_base_perf_results(
    model_key=model_key,
    milestones=eval_milestones,
    n_runs=n_runs,
    judge_repeats=judge_repeats,
    base_seed=base_seed,
    result_names=result_names,
    run_indices=run_indices,
    log_path=LOG_PATH,
    overwrite=False,
)

## Run Worker

In [ ]:
from src.evaluate.judge import score_pending_unscored

await score_pending_unscored(
    model_key=model_key,
    milestones=eval_milestones,
    n_runs=n_runs,
    judge_repeats=judge_repeats,
    judge_config=judge_config,
    methods=editing_methods,
    result_names=result_names,
    run_indices=run_indices,
    **worker_config,
)

## Rebuild Perf From Scored

In [ ]:
from src.evaluate.judge import rebuild_perf_from_scored

rebuild_perf_from_scored(
    model_key=model_key,
    milestones=eval_milestones,
    n_runs=n_runs,
    judge_repeats=judge_repeats,
    methods=editing_methods,
    result_names=result_names,
    run_indices=run_indices,
    log_path=LOG_PATH,
)

## Notes

- `file_concurrency` controls how many milestone files are scored at once.
- `judge_config["max_concurrency"]` controls how many judge API requests each file can launch internally.
- Total judge pressure is roughly `file_concurrency * judge_config["max_concurrency"]`.
- The worker is idempotent: existing `scored/` files are skipped unless `overwrite=True`.
- Run the base perf and rebuild perf cells explicitly when those artifacts should be refreshed.